In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils 
from torchvision.datasets import ImageFolder
import torchvision.models as models
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import gc
import os

In [41]:
# Calculates first order si
def first_order_si(data, labels):
    device = data.device
    labels = labels.reshape((-1,1))
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=2, dim=0, largest=False)
    homolabels = (labels[indices[1]] == labels)
    si = homolabels.sum() / data.shape[0]
    return si

# Calculates high order si (order=2)
def high_order_si(data, labels):
    device = data.device
    labels = labels.reshape((-1,1))
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=3, dim=0, largest=False)
    homolabels = (labels[indices[1]] == labels) * (labels[indices[2]] == labels)
    si = homolabels.sum() / data.shape[0]
    return si

# Calculates high order soft si (order=2)
def high_order_soft_si(data, labels):
    device = data.device
    labels = labels.reshape((-1,1))
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=3, dim=0, largest=False)
    homolabels = (labels[indices[1]] == labels).sum(1) + (labels[indices[2]] == labels).sum(1)
    si = homolabels.sum() / 2 / data.shape[0]
    return si

# Calculates center based si
def center_based_si(data, labels):
    device = data.device
    labels = labels.reshape((-1,1))
    kinds_of_label = torch.unique(labels)
    centers = torch.zeros((kinds_of_label.shape[0], data.shape[1]), device=device)
    for i, label in enumerate(kinds_of_label):
        centers[i] = data[labels[:, 0] == label].mean(0)
        # print(centers[i])
    mat = torch.cdist(data, centers)
    # print('cdist:\n', mat)
    _, indices = torch.topk(mat, k=1, dim=1, largest=False)
    # print(indices)
    # print(kinds_of_label)
    # print(kinds_of_label[indices])
    # print(labels)
    homolabels = (kinds_of_label[indices] == labels)
    # print(homolabels)
    si = homolabels.sum() / data.shape[0]
    return si

# Calculates anti si (order=2)
def anti_si(data, labels):
    device = data.device
    labels = labels.reshape((-1,1))
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=3, dim=0, largest=False)
    homolabels = (labels[indices[1]] != labels) * (labels[indices[2]] != labels)
    si = homolabels.sum() / data.shape[0]
    return si

In [62]:
device = 'cuda'
num_of_data = 100
num_of_classes = 2
data = torch.rand((num_of_data,1), device=device)
labels = torch.randint(0 ,num_of_classes ,(1, num_of_data, 1, 1), device=device)
first_order_si(data, labels), high_order_si(data, labels), center_based_si(data, labels), anti_si(data, labels)

(tensor(0.5000, device='cuda:0'),
 tensor(0.2000, device='cuda:0'),
 tensor(0.5300, device='cuda:0'),
 tensor(0.2500, device='cuda:0'))

In [272]:
device = 'cuda'
num_of_data = 100000
num_of_classes = 4
data = torch.rand((num_of_data,100), device=device)
labels = torch.randint(0 ,num_of_classes ,(1, num_of_data, 1, 1), device=device)

device = data.device
labels = labels.reshape((-1,1))
kinds_of_label = torch.unique(labels)
centers = torch.zeros((kinds_of_label.shape[0], data.shape[1]), device=device)
for i, label in enumerate(kinds_of_label):
    centers[i] = data[labels[:, 0] == label].mean(0)
    # print(centers[i])
mat = torch.cdist(data, centers)
# print('cdist:\n', mat)
_, indices = torch.topk(mat, k=1, dim=1, largest=False)
# print(indices)
# print(kinds_of_label)
# print(kinds_of_label[indices])
homolabels = (kinds_of_label[indices] == labels)
# print(homolabels)
si = homolabels.sum() / data.shape[0]
print(si)

tensor(0.2636, device='cuda:0')


In [3]:
data = torch.tensor([
    [1],
    [1.1],
    [1.2],
    [1.15],
    # [2.1],
    # [1.2],
    # [3],
    # [3.1],
    # [3.2],
])
labels = torch.tensor([
    [1],
    [1],
    [1],
    [2],
    # [2],
    # [2],
    # [3],
    # [3],
    # [3],
])
first_order_si(data, labels), high_order_si(data, labels), center_based_si(data, labels), anti_si(data, labels)

(tensor(0.2500), tensor(0.), tensor(0.7500), tensor(0.2500))

In [48]:
first_order_si(data, labels)

tensor(0.1370, device='cuda:0')

In [41]:
high_order_si(data, labels)

tensor(0.0110, device='cuda:0')

In [42]:
center_based_si(data, labels)

kinds_of_label:  tensor([0, 1, 2, 3, 4, 5, 6, 7], device='cuda:0')
labels.shape: torch.Size([1000, 1])
label: tensor(0, device='cuda:0')
label: tensor(1, device='cuda:0')
label: tensor(2, device='cuda:0')
label: tensor(3, device='cuda:0')
label: tensor(4, device='cuda:0')
label: tensor(5, device='cuda:0')
label: tensor(6, device='cuda:0')
label: tensor(7, device='cuda:0')


tensor(0.4760, device='cuda:0')

In [43]:
anti_si(data, labels)

tensor(0.7930, device='cuda:0')

In [83]:
# Calculates low order and high order hard si. order = 1, 2, 3,...
def si(data, label, order=1):
    device = data.device
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=order+1, dim=0, largest=False)
    homolabel = (label[indices[1]] == label)
    if order == 1:
        si = homolabel.sum() / data.shape[0]
        return si
    else:  
        for i in range(order-1):
            homolabel *= (label[indices[i+2]] == label)
    si = homolabel.sum() / data.shape[0]
    return si

In [87]:
# Calculates soft si for any order. order = 2, 3,...
def soft_si(data, label, order=2):
    device = data.device
    mat = torch.cdist(data, data)
    _, indices = torch.topk(mat, k=order+1, dim=0, largest=False)
    homolabel = (label[indices[1]] == label).sum(1)
    for i in range(order-1):
        homolabel += (label[indices[i+2]] == label).sum(1)
    si = homolabel.sum() / order / data.shape[0]
    return si

In [93]:
torch.tensor([[1,2,3],
              [8,5,6]]).float().mean(0)

tensor([4.5000, 3.5000, 4.5000])

In [80]:
torch.tensor([True,True,True]) + torch.tensor([False,False,False])
torch.tensor([[True,True,True]]).sum(0).float().mean(-1)

tensor(1.)

In [94]:
torch.unique(torch.tensor([[1,2,3]]))

tensor([1, 2, 3])

In [96]:
torch.unique(torch.tensor([[1],[2],[3]]))

tensor([1, 2, 3])

In [90]:
0.5*3/4

0.375

In [ ]:
torch.min()

In [ ]:
data.min()

In [21]:
import torch

# Example input tensor
x = torch.tensor([[0, 2, 1], [1, 4, 2]], dtype=torch.float32)

# Get the indices of the two smallest values in each column
k = 1
_, indices = torch.topk(x, k=k, dim=0, largest=False)

print(indices[0])

tensor([0, 0, 0])


In [ ]:
def high_si(data, label):
    device = data.device
    mat = torch.cdist(data, data) + 1e6*torch.eye(data.shape[0], device=device)
    a,b = mat.min(0)
    si = (label[b] == label).sum() / data.shape[0]
    return si